# 07 — VideoMAE V2 + RGB Transformer baseline (demo ASL Citizen top-50)

This is the **RGB-only research baseline**. It selects the highest-ranked ASL-LEX words and preserves the original ASL Citizen `train` / `validation` / `test` labels. No clip or signer is reassigned. Videos are cached persistently on Drive and reused across runtimes; this baseline does not use pose, a graph encoder, or feature fusion.

VideoMAE V2 stays frozen. Only the projection, a compact **64-dimensional RGB Temporal Transformer (1 layer, 2 heads)**, and the classifier are trained. The run may train for at most **100 epochs**, but early stopping ends it after validation loss stops improving; final validation and test results are produced from `best_checkpoint.pt`. Macro-F1 is the unweighted mean of per-word F1 scores. `MAX_TRAIN_BATCHES=0` and `MAX_EVAL_BATCHES=0` keep every available clip in each official split. The backbone is [OpenGVLab/VideoMAEv2-Base](https://huggingface.co/OpenGVLab/VideoMAEv2-Base), licensed CC BY-NC 4.0.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path

PROJECT_GIT_URL = 'https://github.com/stillthethrone/silent-signal.git'
PROJECT_GIT_REF = 'feat/asl-citizen-videomaev2-demo-baseline'  # @param {type:'string'}
RESULTS_ROOT_STR = '/content/drive/.shortcut-targets-by-id/1-oYEcvJh4ylv_f4AKkJkCjs3FgzjDBFE/silent-signal-results/asl_citizen'  # @param {type:'string'}
PERSIST_VIDEO_CACHE = True  # @param {type:'boolean'}
STREAM_EXTRACT_SELECTED = True  # @param {type:'boolean'}
ACCEPT_ASL_CITIZEN_LICENSE = False  # @param {type:'boolean'}
MODEL_ID = 'OpenGVLab/VideoMAEv2-Base'  # @param {type:'string'}
MODEL_REVISION = '0e826d7e85e39f9d951e331cd91c5c2d8142d385'  # @param {type:'string'}
BASELINE_RUN_NAME = 'videomaev2_rgb_transformer_demo50_official_compact64_v1'  # @param {type:'string'}
CLASS_COUNT = 50  # @param {type:'integer'}
MAX_EPOCHS = 100  # @param {type:'integer'}
BATCH_SIZE = 2  # @param {type:'integer'}
MAX_TRAIN_BATCHES = 0  # @param {type:'integer'}
MAX_EVAL_BATCHES = 0  # @param {type:'integer'}
RGB_EMBEDDING_DIM = 64  # @param {type:'integer'}
RGB_LAYERS = 1  # @param {type:'integer'}
RGB_HEADS = 2  # @param {type:'integer'}
RGB_DROPOUT = 0.5  # @param {type:'number'}
LABEL_SMOOTHING = 0.15  # @param {type:'number'}
WEIGHT_DECAY = 0.04  # @param {type:'number'}
RANDOM_CROP_SCALE_MIN = 0.85  # @param {type:'number'}
COLOR_JITTER = 0.1  # @param {type:'number'}
GRADIENT_CLIP_NORM = 1.0  # @param {type:'number'}
CHECKPOINT_EVERY = 20  # @param {type:'integer'}
EARLY_STOPPING_PATIENCE = 10  # @param {type:'integer'}
EARLY_STOPPING_MIN_DELTA = 0.001  # @param {type:'number'}
PROGRESS_EVERY = 5  # @param {type:'integer'}
LEARNING_RATE = 0.0003  # @param {type:'number'}
NUM_WORKERS = 2  # @param {type:'integer'}
DEVICE = 'auto'  # @param ['auto', 'cuda', 'cpu']
RESUME = False  # @param {type:'boolean'}
RUN_TEST = True  # Evaluate the untouched official test split only after restoring the best checkpoint.
RUN_TRAINING = True  # @param {type:'boolean'}

if CLASS_COUNT != 50:
    raise ValueError('Notebook demo này được cố định ở đúng 50 từ.')
PROJECT_ROOT = Path('/content/silent-signal')
RESULTS_ROOT = Path(RESULTS_ROOT_STR)
TOP200_ROOT = RESULTS_ROOT / 'subsets/asl_citizen_asllex_top200'
# Giữ đường dẫn cache top-30 cũ để tái sử dụng video đã tải; cell cache chỉ tải thêm clip còn thiếu cho top-50.
DRIVE_DATASET_ROOT = TOP200_ROOT / 'datasets/asl_citizen_top30'
LOCAL_DATASET_ROOT = Path('/content/ASL_Citizen')
DATASET_ROOT = DRIVE_DATASET_ROOT if PERSIST_VIDEO_CACHE else LOCAL_DATASET_ROOT
SOURCE_MANIFEST = TOP200_ROOT / 'manifest.csv'
SOURCE_SELECTION = TOP200_ROOT / 'selection_report.json'
BASELINE_ROOT = TOP200_ROOT / 'baselines' / BASELINE_RUN_NAME

## Sơ đồ baseline và ranh giới thí nghiệm

```text
ASL Citizen video ─► 16 RGB frames ─► VideoMAE V2 [FROZEN] ─► 8 temporal tokens
                                                               │
                                classifier 50 lớp ◄─ RGB Transformer [TRAINABLE]

The original ASL Citizen split is used; there is no pose, graph encoder, or fusion.
```

`validation` selects the checkpoint. The untouched official `test` split is evaluated once after the best checkpoint has been restored.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
fig, ax = plt.subplots(figsize=(13, 3))
boxes = [('Official RGB video', 0.01, '#457b9d'), ('16 frames\n224×224', 0.21, '#2a9d8f'), ('VideoMAE V2\nFROZEN', 0.41, '#6c757d'), ('RGB Transformer\n64-d, TRAINABLE', 0.61, '#e9c46a'), ('Classifier\nASL word', 0.81, '#e76f51')]
for label, x, color in boxes:
    ax.add_patch(FancyBboxPatch((x, 0.35), 0.15, 0.32, boxstyle='round,pad=0.02', facecolor=color, alpha=0.9))
    ax.text(x + 0.075, 0.51, label, ha='center', va='center', color='white', fontsize=10, weight='bold')
for x in (0.16, 0.36, 0.56, 0.76):
    ax.add_patch(FancyArrowPatch((x, 0.51), (x + 0.045, 0.51), arrowstyle='-|>', mutation_scale=18))
ax.text(0.5, 0.12, 'Official ASL Citizen split; validation selects the checkpoint and test stays untouched', ha='center', color='#9d0208', weight='bold')
ax.set(xlim=(0, 1), ylim=(0, 1)); ax.axis('off'); plt.show()

## Lấy đúng nhánh và cài môi trường Colab

In [ ]:
import os, shutil, subprocess, sys, time
# Pipeline này chỉ dùng PyTorch; không để Transformers probe TensorFlow/JAX của Colab.
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['USE_JAX'] = '0'
os.environ['TRANSFORMERS_NO_TF'] = '1'
def run(command):
    command = list(map(str, command))
    started = time.perf_counter()
    print(time.strftime('[%H:%M:%S] START'), ' '.join(command), flush=True)
    subprocess.run(command, check=True)
    print(time.strftime('[%H:%M:%S] DONE '), f'{time.perf_counter() - started:.1f}s', flush=True)

if not PROJECT_ROOT.exists():
    run(['git', 'clone', '--branch', PROJECT_GIT_REF, '--single-branch', PROJECT_GIT_URL, PROJECT_ROOT])
else:
    run(['git', '-C', PROJECT_ROOT, 'fetch', 'origin', PROJECT_GIT_REF])
    run(['git', '-C', PROJECT_ROOT, 'checkout', PROJECT_GIT_REF])
    run(['git', '-C', PROJECT_ROOT, 'pull', '--ff-only', 'origin', PROJECT_GIT_REF])
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', PROJECT_ROOT])
# Editable installs performed in a subprocess do not refresh the running Colab kernel's sys.path.
# Register src explicitly before later cells import helpers from silent_signal.
PROJECT_SRC = str(PROJECT_ROOT / 'src')
if PROJECT_SRC not in sys.path:
    sys.path.insert(0, PROJECT_SRC)
run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy==2.1.3', 'transformers==4.48.3', 'timm==1.0.15', 'easydict==1.13', 'opencv-python-headless==4.10.0.84', 'matplotlib==3.10.0'])
run([sys.executable, '-c', "import numpy; from transformers import PreTrainedModel; assert hasattr(numpy.dtypes, 'StringDType'); print('Import check PASS | NumPy', numpy.__version__)"])
import silent_signal, torch
print('Project import PASS:', silent_signal.__file__)
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))

## Chốt 50 từ và kiểm tra split trước khi tải video

The selected words are the highest-ranked `SignFrequency(M)` entries with clips in all three official splits; they are not selected by clip count. The notebook keeps every selected row's original ASL Citizen split label. The CLI writes `all.csv`, `train.csv`, `validation.csv`, and `test.csv` manifests to Drive so the exact comparison set is auditable.

In [ ]:
import csv, json
from collections import Counter
if not SOURCE_MANIFEST.is_file(): raise FileNotFoundError(SOURCE_MANIFEST)
if not SOURCE_SELECTION.is_file(): raise FileNotFoundError(SOURCE_SELECTION)
selection = json.loads(SOURCE_SELECTION.read_text(encoding='utf-8'))
with SOURCE_MANIFEST.open(encoding='utf-8-sig', newline='') as handle:
    all_rows = list(csv.DictReader(handle))
all_counts = Counter((int(row['class_index']), row['split']) for row in all_rows)
required_splits = ('train', 'validation', 'test')
eligible = [item for item in selection['classes'] if all(all_counts[item['subset_class_index'], split] > 0 for split in required_splits)]
classes = eligible[:CLASS_COUNT]
if len(classes) != CLASS_COUNT:
    raise RuntimeError(f'Chỉ có {len(classes)} lớp xếp hạng có đủ cả ba official split.')
if [item['rank'] for item in classes] != sorted(item['rank'] for item in classes):
    raise RuntimeError('Thứ tự ASL-LEX rank không tăng dần.')
selected_indices = {str(item['subset_class_index']) for item in classes}
# Preserve the official split label carried by every ASL Citizen manifest row.
rows = [row for row in all_rows if row['class_index'] in selected_indices]
ids = {split: {row['sample_id'] for row in rows if row['split'] == split} for split in ('train', 'validation', 'test')}
if ids['train'] & ids['validation'] or ids['train'] & ids['test'] or ids['validation'] & ids['test']:
    raise RuntimeError('LEAKAGE: sample_id xuất hiện ở nhiều split.')
if sum(map(len, ids.values())) != len(rows): raise RuntimeError('Split thiếu hoặc sample_id trùng.')
signers = {split: {row['signer_id'] for row in rows if row['split'] == split} for split in ids}
if signers['train'] & signers['validation'] or signers['train'] & signers['test'] or signers['validation'] & signers['test']:
    raise RuntimeError('LEAKAGE: signer xuất hiện ở nhiều split.')
counts = Counter((int(row['class_index']), row['split']) for row in rows)
print(f"{'rank':>4}  {'gloss':<25} {'train':>13} {'validation':>13} {'test':>13}")
for item in classes:
    index = item['subset_class_index']; total = sum(counts[index, split] for split in ids)
    values = [f"{counts[index, split]} ({100 * counts[index, split] / total:.1f}%)" for split in ('train', 'validation', 'test')]
    print(f"{item['rank']:>4}  {item['gloss_name'][:25]:<25} {values[0]:>13} {values[1]:>13} {values[2]:>13}")
print('\nOfficial split isolation: PASS | clips:', {split: len(value) for split, value in ids.items()}, '| signers:', {split: len(value) for split, value in signers.items()})

## Stream-extract và cache bền vững đầy đủ video của 50 từ

This cell reads only the required members from the official Microsoft archive and does not store the full archive. With `PERSIST_VIDEO_CACHE=True`, it reuses the existing cache at `datasets/asl_citizen_top30`, verifies every path, and extracts only missing videos. Each video remains stored once at its original relative path.

In [ ]:
import sys, time
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
selected_video_paths = {row['video_path'] for row in rows}
missing = sorted(path for path in selected_video_paths if not (DATASET_ROOT / path).is_file())
print(f'Video hiện có: {len(selected_video_paths) - len(missing)}/{len(selected_video_paths)}; thiếu: {len(missing)}', flush=True)
if missing:
    if not STREAM_EXTRACT_SELECTED:
        raise FileNotFoundError('Thiếu video và STREAM_EXTRACT_SELECTED=False.')
    if not ACCEPT_ASL_CITIZEN_LICENSE:
        raise RuntimeError('Đọc điều khoản Microsoft rồi bật ACCEPT_ASL_CITIZEN_LICENSE=True.')
    from silent_signal.data.asl_download import extract_remote_archive
    extract_remote_archive(DATASET_ROOT, include_paths=selected_video_paths)
remaining = [path for path in selected_video_paths if not (DATASET_ROOT / path).is_file()]
if remaining: raise FileNotFoundError(f'Vẫn thiếu {len(remaining)} video sau extract.')
cache_bytes = sum((DATASET_ROOT / path).stat().st_size for path in selected_video_paths)
BASELINE_ROOT.mkdir(parents=True, exist_ok=True)
cache_report = {
    'schema_version': 1, 'state': 'ready', 'persistent': PERSIST_VIDEO_CACHE,
    'dataset_root': str(DATASET_ROOT), 'selected_videos': len(selected_video_paths),
    'size_bytes': cache_bytes, 'size_gib': round(cache_bytes / 1024**3, 3),
    'split_policy': 'official ASL Citizen train/validation/test; never re-split',
    'split_counts': {split: len(ids[split]) for split in ('train', 'validation', 'test')},
    'updated_unix': time.time(),
}
cache_report_path = BASELINE_ROOT / 'persistent_video_cache.json'
temporary_report = cache_report_path.with_suffix('.json.tmp')
temporary_report.write_text(json.dumps(cache_report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
temporary_report.replace(cache_report_path)
print(f'Dataset demo sẵn sàng: {len(selected_video_paths)}/{len(selected_video_paths)} video', flush=True)
print(f'Cache: {DATASET_ROOT} | {cache_bytes / 1024**3:.2f} GiB | persistent={PERSIST_VIDEO_CACHE}', flush=True)

## Train có log, giới hạn và resume

Batch logs show progress, samples, loss, top-1, and ETA. Every epoch also records train/validation Macro-F1 in `history.json`, `baseline_report.json`, and `training_curves.png`. `MAX_EPOCHS=100` is only an upper bound: early stopping uses validation loss with patience 10 and restores `best_checkpoint.pt`. The compact head matches the stable earlier setup: embedding 64, one Transformer layer, two attention heads, dropout 0.5, weight decay 0.04, and label smoothing 0.15. The new run name prevents overwriting earlier checkpoints, while the video cache is reused. Keep `RESUME=False` on the first run; enable it only after a disconnect in this same run.

In [ ]:
def run_stream(command):
    command = list(map(str, command))
    print('+', ' '.join(command), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout: print(line, end='', flush=True)
    code = process.wait()
    if code: raise subprocess.CalledProcessError(code, command)

command = [sys.executable, '-u', '-m', 'silent_signal.cli.train_videomaev2_demo',
    '--manifest', SOURCE_MANIFEST, '--selection-report', SOURCE_SELECTION,
    '--dataset-root', DATASET_ROOT, '--output-root', BASELINE_ROOT,
    '--model-id', MODEL_ID, '--model-revision', MODEL_REVISION,
    '--classes', CLASS_COUNT, '--epochs', MAX_EPOCHS, '--batch-size', BATCH_SIZE,
    '--learning-rate', LEARNING_RATE, '--weight-decay', WEIGHT_DECAY,
    '--label-smoothing', LABEL_SMOOTHING, '--max-train-batches', MAX_TRAIN_BATCHES,
    '--max-eval-batches', MAX_EVAL_BATCHES, '--checkpoint-every', CHECKPOINT_EVERY,
    '--early-stopping-patience', EARLY_STOPPING_PATIENCE,
    '--early-stopping-min-delta', EARLY_STOPPING_MIN_DELTA,
    '--progress-every', PROGRESS_EVERY, '--num-workers', NUM_WORKERS, '--device', DEVICE,
    '--rgb-embedding-dim', RGB_EMBEDDING_DIM, '--rgb-layers', RGB_LAYERS,
    '--rgb-heads', RGB_HEADS, '--rgb-dropout', RGB_DROPOUT,
    '--random-crop-scale-min', RANDOM_CROP_SCALE_MIN, '--color-jitter', COLOR_JITTER,
    '--gradient-clip-norm', GRADIENT_CLIP_NORM]
if RESUME: command.append('--resume')
if RUN_TEST: command.append('--run-test')
if RUN_TRAINING: run_stream(command)
else: print('RUN_TRAINING=False — chỉ kiểm tra dữ liệu, chưa train.')

## Kết quả bền vững trên Drive

In [ ]:
from IPython.display import Image, display
report_path = BASELINE_ROOT / 'baseline_report.json'
if report_path.is_file():
    report = json.loads(report_path.read_text(encoding='utf-8'))
    print(json.dumps(report, ensure_ascii=False, indent=2))
    curve = BASELINE_ROOT / 'training_curves.png'
    if curve.is_file(): display(Image(filename=str(curve)))
    print('PASS demo baseline. Tiếp theo chạy notebook 08 trên validation_predictions.csv.')
else:
    print('Chưa có baseline_report.json. Nếu vừa dừng giữa chừng, checkpoint vẫn ở:', BASELINE_ROOT / 'last_checkpoint.pt')